# CNN-BiLSTM EEG ADHD Prediction Pipeline
**Run all cells in order. GPU T4 x1 must be enabled. Internet must be ON.**

## Step 1 — Clone the pipeline code from GitHub

In [ ]:
import os

# Remove old clone if re-running
if os.path.exists("/kaggle/working/project"):
    import shutil
    shutil.rmtree("/kaggle/working/project")

!git clone https://github.com/sathishshah/ADHD-Prediction.git /kaggle/working/project
print("Clone done.")

## Step 2 — Install only missing packages
Kaggle already has TensorFlow, numpy, scipy, sklearn, pandas, matplotlib installed.
We only install what is missing to avoid breaking the pre-installed environment.

In [ ]:
# Only install packages not pre-installed on Kaggle
# Do NOT reinstall tensorflow or numpy — Kaggle's versions are already correct
!pip install -q "shap>=0.44" mne
print("Done installing shap and mne.")

## Step 3 — Diagnose what is in /kaggle/input
This cell lists everything Kaggle has mounted. Use this to confirm the dataset is there.

In [ ]:
import os

print("=" * 60)
print("ALL folders and files under /kaggle/input:")
print("=" * 60)

mat_count = 0
mat_dirs  = set()

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (depth + 1)
    for f in files[:5]:  # show first 5 files per folder
        print(f"{sub_indent}{f}")
        if f.endswith(".mat"):
            mat_count += 1
            mat_dirs.add(root)
    if len(files) > 5:
        mat_here = sum(1 for f in files if f.endswith(".mat"))
        mat_count += mat_here - min(mat_here, 5)
        print(f"{sub_indent}... and {len(files)-5} more files ({mat_here} .mat total in this folder)")

print()
print("=" * 60)
if mat_count == 0:
    print("NO .mat files found anywhere.")
    print()
    print("ACTION NEEDED:")
    print("  1. Click '+ Add Data' on the right sidebar")
    print("  2. Search for: EEG data ADHD control children")
    print("  3. Look for a dataset with 121 files (61 ADHD + 60 Control)")
    print("  4. Add it, then re-run this cell")
else:
    print(f"Found {mat_count} .mat files in these folders:")
    for d in sorted(mat_dirs):
        n = sum(1 for f in os.listdir(d) if f.endswith(".mat"))
        print(f"  {d}  ({n} files)")

## Step 4 — Set the data directory
**Only run this after Step 3 shows .mat files.**  
Set `DATA_DIR` to the folder that is the **parent** of the ADHD and Control subfolders.

In [ ]:
# ── SET THIS based on Step 3 output ──────────────────────────────────────────
# Example: if Step 3 shows .mat files at /kaggle/input/eeg-dataset-for-adhd/ADHD/
# then DATA_DIR = "/kaggle/input/eeg-dataset-for-adhd"
DATA_DIR = "/kaggle/input/eeg-dataset-for-adhd"
# ─────────────────────────────────────────────────────────────────────────────

OUT_DIR = "/kaggle/working/results"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)

# Verify the path exists
if os.path.isdir(DATA_DIR):
    print(f"OK  DATA_DIR exists: {DATA_DIR}")
    contents = os.listdir(DATA_DIR)
    print(f"    Contents: {contents}")
else:
    print(f"ERROR: {DATA_DIR} does not exist. Update DATA_DIR above based on Step 3.")

print(f"OUT_DIR = {OUT_DIR}")

## Step 5 — Verify TensorFlow works

In [ ]:
import tensorflow as tf
import numpy as np
import shap
import mne
import scipy
import sklearn

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"SHAP       : {shap.__version__}")
print(f"MNE        : {mne.__version__}")
print(f"SciPy      : {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPUs available: {len(gpus)}")
for g in gpus:
    print(f"  {g}")
if not gpus:
    print("WARNING: No GPU found. Enable GPU T4 in the right sidebar > Accelerator.")

## Step 6 — Smoke test (~10 minutes)
Always run this before the full run to confirm the pipeline works end-to-end.

In [ ]:
!cd /kaggle/working/project && python -m adhd_pipeline.run_all \
    --data-dir "$DATA_DIR" \
    --out-dir /kaggle/working/results_quick \
    --quick \
    --no-shap

## Step 7 — Full pipeline run (~90 minutes)
**Only run after Step 6 finishes with no errors.**  
`--checkpoint` saves each fold so a disconnect does not lose progress.

In [ ]:
!cd /kaggle/working/project && python -m adhd_pipeline.run_all \
    --data-dir "$DATA_DIR" \
    --out-dir "$OUT_DIR" \
    --checkpoint

## Step 8 — Check output files

In [ ]:
import json

expected = [
    "all_results.json",
    "macro_replacements.tex",
    "folds.json",
    "shap_channel_importance.npy",
    "shap_band_importance.npy",
    "figures/leakage_bars.csv",
    "figures/baseline_bars.csv",
    "figures/roc_aucs.csv",
    "figures/confusion_matrix.csv",
    "figures/shap_channels.csv",
]

print("Output file check:\n")
all_ok = True
for fname in expected:
    path = os.path.join(OUT_DIR, fname)
    exists = os.path.isfile(path)
    status = "OK     " if exists else "MISSING"
    print(f"  [{status}]  {fname}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("All files present. Proceed to Step 9.")
else:
    print("Some files are missing. Scroll up to check Step 7 for error messages.")

tex_path = os.path.join(OUT_DIR, "macro_replacements.tex")
if os.path.isfile(tex_path):
    print("\n" + "=" * 60)
    print("macro_replacements.tex  (paste this block into main.tex):")
    print("=" * 60)
    with open(tex_path) as f:
        print(f.read())

## Step 9 — Print accuracy summary

In [ ]:
results_path = os.path.join(OUT_DIR, "all_results.json")
if not os.path.isfile(results_path):
    print("all_results.json not found. Run Step 7 first.")
else:
    with open(results_path) as f:
        results = json.load(f)

    model_order = ["LogisticRegression", "SVM", "RandomForest",
                   "CNN_only", "LSTM_only", "CNN_BiLSTM"]

    for protocol_key, label in [
        ("protocol_a", "PROTOCOL A  Segment-wise (leaky)"),
        ("protocol_b", "PROTOCOL B  Subject-wise (clean)"),
        ("loso",       "PROTOCOL    LOSO"),
    ]:
        if protocol_key not in results:
            continue
        print(f"\n{label}")
        print("-" * 55)
        print(f"  {'Model':<22}  {'Acc mean±std':>16}   {'AUC':>5}")
        print("-" * 55)
        for model in model_order:
            if model not in results[protocol_key]:
                continue
            s = results[protocol_key][model]["summary"]
            acc_mean, acc_std = s["accuracy"]
            auc_mean, _      = s["roc_auc"]
            print(f"  {model:<22}  {acc_mean:>6.1f} ± {acc_std:<5.1f}  {auc_mean:.3f}")

    if results.get("significance"):
        sig = results["significance"]
        print("\nLEAKAGE GAP (CNN-BiLSTM  A vs B)")
        print("-" * 40)
        for k, v in sig.items():
            print(f"  {k}: {v}")

## Step 10 — Zip and download all results

In [ ]:
import shutil

zip_path = "/kaggle/working/results_export"
shutil.make_archive(zip_path, "zip", OUT_DIR)
size_mb = os.path.getsize(zip_path + ".zip") / 1e6
print(f"Created: {zip_path}.zip  ({size_mb:.1f} MB)")
print()
print("To download:")
print("  1. Click the Output tab on the right sidebar")
print("  2. Download results_export.zip")
print("  3. Extract -> macro_replacements.tex and figures/*.csv are inside")